# **Разведочный анализ данных**

По истории действий клиента предскажите, на предложение какого продукта он нажмёт следующим.

Во входных данных — последовательности просмотров, кликов и других событий. Для обучающих примеров известен ответ — код продукта. Для тестовых примеров его нужно предсказать.

Нужно выбрать один продукт. Клики учитываются во всех точках показа: разные баннеры и варианты предложения одного продукта относятся к одному классу.

train — 100 000 клиентов, test — 12 359; у каждого история до 2 000 событий: время, код действия, продукт

Посмотрим на события

In [1]:
import polars as pl

DATA = '../data'
train = pl.read_parquet(f'{DATA}/train.parquet')
test = pl.read_parquet(f'{DATA}/test.parquet')
ed = pl.read_parquet(f'{DATA}/event_dictionary.parquet')

def flatten(df):
    # список событий в каждой строке -> одна строка на событие
    return (df.select('example_id', 'events')
              .explode('events')          # раскрыть список в строки
              .unnest('events')           # struct -> 3 обычных столбца
              .join(ed, on='encoded_event', how='left'))  # добавить тип события
              

ftr = flatten(train)
fte = flatten(test)
ftr.head()

example_id,timestamp,encoded_event,encoded_product,event_type,is_target_event
str,i64,i32,i32,str,bool
"""3068746855fd6f7346e5b708f7c8d5…",1770478684290,962,null,"""view""",false
"""3068746855fd6f7346e5b708f7c8d5…",1770478685179,13,13,"""view""",false
"""3068746855fd6f7346e5b708f7c8d5…",1770478692945,234,null,"""other""",false
"""3068746855fd6f7346e5b708f7c8d5…",1770478694812,13,58,"""view""",false
"""3068746855fd6f7346e5b708f7c8d5…",1770533802067,962,null,"""view""",false


## 1. Типы событий

Какие бывают события?

In [2]:
event_types_df = ftr.group_by('event_type').agg(pl.len())
event_types_df

event_type,len
str,u32
"""click""",3064857
"""other""",25420421
"""view""",31799403
"""session_boundary""",13413839


В процентном соотношении:

In [3]:
event_types_df.with_columns((pl.col('len') / pl.col('len').sum() * 100).round(1).alias('share_%')).sort('len', descending=True)

event_type,len,share_%
str,u32,f64
"""view""",31799403,43.1
"""other""",25420421,34.5
"""session_boundary""",13413839,18.2
"""click""",3064857,4.2


Вывод: посмотрели на данные, выяснили, что кликов всего около 4 процентов.

## 2. Показы и клики

Какой код означает целевой клик?

In [4]:
ed.filter(pl.col('is_target_event'))

encoded_event,event_type,is_target_event
i32,str,bool
469,"""click""",true


У каких событий есть продукт?

In [5]:
events_with_product_df = (ftr.filter(pl.col('encoded_product').is_not_null())
    .group_by('encoded_event', 'event_type')
    .agg(pl.len().alias('n'),
         pl.col('encoded_product').n_unique().alias('n_products'))
    .sort('n', descending=True)
    )

events_with_product_df.head(10)

encoded_event,event_type,n,n_products
i32,str,u32,u32
13,"""view""",13238060,41
399,"""view""",519489,38
542,"""other""",342074,40
835,"""click""",164718,28
469,"""click""",111743,42
666,"""view""",79412,3
194,"""other""",54125,17
459,"""click""",50010,3
162,"""view""",24569,2


| столбец | что значит |
|---|---|
| `encoded_event` | код действия |
| `event_type` | тип действия из справочника |
| `n` | сколько раз действие с продуктом встретилось во всех историях train |
| `n_products` | сколько **разных** продуктов встречается у этого действия |

Вывод: событие `13` — показ баннера продукта: это самое частое действие с продуктом (13,2 млн) и затрагивает 41 продукт. Событие `469` — целевой клик по баннеру (по справочнику), в истории это прошлые клики клиента: 112 тыс. Кликают примерно по одному баннеру из ста показанных.

## 3. Простые правила

А что если всегда отвечать самым частым продуктом?

In [6]:
train['target_product'].value_counts(normalize=True).sort('proportion', descending=True).head(5)

target_product,proportion
i32,f64
55,0.16008
6,0.1483
35,0.1408
63,0.10438
58,0.07249


Accuracy - 0.16 для продукта 55

Попробуем следующее: для каждого клиента берём продукт из его последнего события 13 и проверяем, совпал ли он с ответом.

In [7]:
last_view = (ftr.filter(pl.col('encoded_event') == 13)       # только показы баннеров
                .group_by('example_id')
                .agg(pl.col('encoded_product').last().alias('last_view')))  # последний по времени

rules = train.select('example_id', 'target_product').join(last_view, on='example_id', how='left')
(rules['last_view'] == rules['target_product']).fill_null(False).mean() #столбец из 0;1: считаем долю угаданных

0.34667

Accuracy - 0.34! - более чем в 2 раза лучше 

Теперь попробуем для каждого клиента брать продукт из последнего события 469 и проверять, есть ли совпадение с ответом.

In [8]:
last_click = (ftr.filter(pl.col('encoded_event') == 469)
                .group_by('example_id')
                .agg(pl.col('encoded_product').last().alias('last_click')))

rules = train.select('example_id', 'target_product').join(last_click, on='example_id', how='left')
(rules['last_click'] == rules['target_product']).fill_null(False).mean()

0.1508

In [9]:
rules['last_click'].is_not_null().mean()   # доля клиентов, у которых был хоть один клик 469

0.49051

Клик проигрывает, так как клик 469 есть только у 49% клиентов, для остальных клиентов мы автоматически проиграли и не смогли выбрать продукт.

## 4. Timeshift

Когда заканчиваются истории train и test?

In [10]:
def end_weeks(flat, split):
    return (flat.group_by('example_id')
                .agg(pl.col('timestamp').max())                                    # последнее событие примера
                .with_columns(pl.from_epoch('timestamp', time_unit='ms')           # мс -> дата
                                .dt.truncate('1w').alias('week'),                  # округлить до начала недели
                              pl.lit(split).alias('split')))                       # пометка train/test

weeks = pl.concat([end_weeks(ftr, 'train'), end_weeks(fte, 'test')])

with pl.Config(tbl_rows=-1):          # показать все строки, без "…"
    print(weeks.group_by('week', 'split').len()
               .pivot(on='split', index='week', values='len')   # train и test -> отдельные столбцы
               .sort('week'))

shape: (14, 3)
┌─────────────────────┬───────┬──────┐
│ week                ┆ train ┆ test │
│ ---                 ┆ ---   ┆ ---  │
│ datetime[μs]        ┆ u32   ┆ u32  │
╞═════════════════════╪═══════╪══════╡
│ 2026-01-12 00:00:00 ┆ 8     ┆ null │
│ 2026-01-19 00:00:00 ┆ 25    ┆ null │
│ 2026-01-26 00:00:00 ┆ 43    ┆ null │
│ 2026-02-02 00:00:00 ┆ 126   ┆ null │
│ 2026-02-09 00:00:00 ┆ 270   ┆ null │
│ 2026-02-16 00:00:00 ┆ 613   ┆ null │
│ 2026-02-23 00:00:00 ┆ 6523  ┆ null │
│ 2026-03-02 00:00:00 ┆ 20490 ┆ null │
│ 2026-03-09 00:00:00 ┆ 19302 ┆ 1    │
│ 2026-03-16 00:00:00 ┆ 16371 ┆ null │
│ 2026-03-23 00:00:00 ┆ 16178 ┆ 13   │
│ 2026-03-30 00:00:00 ┆ 18453 ┆ 39   │
│ 2026-04-06 00:00:00 ┆ 1598  ┆ 8133 │
│ 2026-04-13 00:00:00 ┆ null  ┆ 4173 │
└─────────────────────┴───────┴──────┘


**Вывод:** test идёт сразу после train по времени (6 апреля и позже в основном), поэтому валидацию строим на последних днях train.

Почему именно последняя неделя, а не случайные 20%. Валидация должна быть похожа на тест. Тест — это будущее относительно train. Если отложить случайных клиентов из всего периода, модель на валидации будет видеть то же время, на котором училась, и оценка выйдет слишком оптимистичной. Отложив последние дни, мы воспроизводим реальную ситуацию: «учились на прошлом, предсказываем будущее».

Что показывали в апреле?

In [11]:
april = pl.datetime(2026, 4, 6)

view_april = (fte.filter((pl.col('encoded_event') == 13) &
                         (pl.from_epoch('timestamp', time_unit='ms') >= april))
                 .group_by('encoded_product').len()
                 .with_columns((pl.col('len') / pl.col('len').sum()).alias('april_view_share'))
                 .select(pl.col('encoded_product').alias('target_product'), 'april_view_share'))

(train['target_product'].value_counts(normalize=True)
    .rename({'proportion': 'train_target_share'})
    .join(view_april, on='target_product', how='left')
    .sort('train_target_share', descending=True)
    .head(10))

target_product,train_target_share,april_view_share
i32,f64,f64
55,0.16008,null
6,0.1483,0.216777
35,0.1408,0.040321
63,0.10438,0.119644
58,0.07249,0.01158
13,0.063,0.095919
18,0.05964,0.132557
4,0.04193,null
60,0.03687,0.054797


**Вывод:** продукт 55 — самый частый ответ в train (16%), но в апреле его баннер не показывали ни разу; то же с продуктом 4 (ещё 4%). Модель, обученная на train как есть, будет часто предсказывать эти продукты, хотя в тестовом периоде кликнуть по ним почти невозможно. Это примерно пятая часть ответов train — такие случаи нужно обрабатывать отдельно.